In [1]:
# ------------------------------
# 1. Import Libraries
# ------------------------------
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

# Gradient Boosting
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings("ignore")

In [2]:
# ------------------------------
# 2. Load Data
# ------------------------------
import pandas as pd
import csv

# Load train and test CSVs safely, ignoring illegal quotes
train = pd.read_csv(
    "train.csv",
    engine='python',          # use Python engine to handle messy quoting
    quoting=csv.QUOTE_NONE,   # ignore quote characters
    on_bad_lines='skip'       # skip lines that cause parsing errors
)

test = pd.read_csv(
    "test.csv",
    engine='python',
    quoting=csv.QUOTE_NONE,
    on_bad_lines='skip'
)

print(f"Train shape: {train.shape}, Test shape: {test.shape}")

Train shape: (30557, 82), Test shape: (16136, 77)


In [3]:
# ------------------------------
# 3. Feature Engineering
# ------------------------------

# Fill missing pollutant values with mean
pollutant_cols = [
    'L3_NO2_tropospheric_NO2_column_number_density',
    'L3_SO2_SO2_column_number_density',
    'L3_CO_CO_column_number_density'
]

train[pollutant_cols] = train[pollutant_cols].fillna(train[pollutant_cols].mean())
test[pollutant_cols] = test[pollutant_cols].fillna(test[pollutant_cols].mean())

# Create pollutant ratios
for df in [train, test]:
    df['NO2_CO_ratio'] = df['L3_NO2_tropospheric_NO2_column_number_density'] / df['L3_CO_CO_column_number_density']
    df['SO2_CO_ratio'] = df['L3_SO2_SO2_column_number_density'] / df['L3_CO_CO_column_number_density']
    df['NO2_SO2_ratio'] = df['L3_NO2_tropospheric_NO2_column_number_density'] / df['L3_SO2_SO2_column_number_density']

# ------------------------------
# Encode day_of_week and day_of_year from Date
# ------------------------------
for df in [train, test]:
    df['Date'] = pd.to_datetime(df['Date'])  # convert to datetime
    df['day_of_week'] = df['Date'].dt.dayofweek  # Monday=0, Sunday=6
    df['day_of_year'] = df['Date'].dt.dayofyear  # 1-365/366

    # Cyclical encoding
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week']/7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week']/7)
    df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year']/365)
    df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year']/365)

In [4]:
# ------------------------------
# Step 1: Ensure all clustering features have no missing values
# ------------------------------
cluster_features = [
    'temperature_2m_above_ground',
    'relative_humidity_2m_above_ground',
    'u_component_of_wind_10m_above_ground',
    'v_component_of_wind_10m_above_ground',
    'L3_NO2_tropospheric_NO2_column_number_density',
    'L3_SO2_SO2_column_number_density',
    'L3_CO_CO_column_number_density',
    'NO2_CO_ratio'
]

# Fill missing values with mean for all clustering features
train[cluster_features] = train[cluster_features].fillna(train[cluster_features].mean())
test[cluster_features] = test[cluster_features].fillna(test[cluster_features].mean())

# ------------------------------
# Step 2: Scale features
# ------------------------------
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled_train_features = scaler.fit_transform(train[cluster_features])
scaled_test_features = scaler.transform(test[cluster_features])

# ------------------------------
# Step 3: Apply K-Means
# ------------------------------
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=5, random_state=42)
train['cluster_label'] = kmeans.fit_predict(scaled_train_features)
test['cluster_label'] = kmeans.predict(scaled_test_features)

print("K-Means clustering applied successfully. Sample labels:")
print(train[['cluster_label']].head())

K-Means clustering applied successfully. Sample labels:
   cluster_label
0              3
1              3
2              3
3              3
4              3


In [5]:
# ------------------------------
# 5. Prepare Features
# ------------------------------
features = [
    'temperature_2m_above_ground',
    'relative_humidity_2m_above_ground',
    'u_component_of_wind_10m_above_ground',
    'v_component_of_wind_10m_above_ground',
    'L3_NO2_tropospheric_NO2_column_number_density',
    'L3_SO2_SO2_column_number_density',
    'L3_CO_CO_column_number_density',
    'NO2_CO_ratio', 'SO2_CO_ratio', 'NO2_SO2_ratio',
    'day_of_week_sin', 'day_of_week_cos',
    'day_of_year_sin', 'day_of_year_cos',
    'cluster_label'
]

X = train[features]
y = train['target']
X_test = test[features]

In [6]:
# ------------------------------
# 6. Train-Validation Split
# ------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
# ------------------------------
# 1. Import libraries
# ------------------------------
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

# ------------------------------
# 2. Prepare cluster feature
# ------------------------------
# Example cluster features (same as before)
cluster_features = [
    'temperature_2m_above_ground',
    'relative_humidity_2m_above_ground',
    'u_component_of_wind_10m_above_ground',
    'v_component_of_wind_10m_above_ground',
    'L3_NO2_tropospheric_NO2_column_number_density',
    'L3_SO2_SO2_column_number_density',
    'L3_CO_CO_column_number_density',
    'NO2_CO_ratio'
]

# Fill missing values in cluster columns
train[cluster_features] = train[cluster_features].fillna(train[cluster_features].mean())
test[cluster_features] = test[cluster_features].fillna(test[cluster_features].mean())

# Scale features and create clusters
scaler = StandardScaler()
scaled_train_features = scaler.fit_transform(train[cluster_features])
scaled_test_features = scaler.transform(test[cluster_features])

kmeans = KMeans(n_clusters=5, random_state=42)
train['cluster_label'] = kmeans.fit_predict(scaled_train_features)
test['cluster_label'] = kmeans.predict(scaled_test_features)

# ------------------------------
# 3. Train/Validation Split
# ------------------------------
train_set, val_set = train_test_split(
    train, test_size=0.2, random_state=42
)

# ------------------------------
# 4. Feature & Target Setup
# ------------------------------
target = 'target'
# You can include clusters as feature
features = [
    'temperature_2m_above_ground',
    'relative_humidity_2m_above_ground',
    'u_component_of_wind_10m_above_ground',
    'v_component_of_wind_10m_above_ground',
    'L3_NO2_tropospheric_NO2_column_number_density',
    'L3_SO2_SO2_column_number_density',
    'L3_CO_CO_column_number_density',
    'NO2_CO_ratio',
    'cluster_label'
]

X_train = train_set[features]
y_train = train_set[target]
X_val = val_set[features]
y_val = val_set[target]
X_test = test[features]

# ------------------------------
# 5a. Train LightGBM
# ------------------------------
lgb_model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    random_state=42
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='rmse',
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(50)]
)

y_pred_lgb = lgb_model.predict(X_val)
rmse_lgb = mean_squared_error(y_val, y_pred_lgb) ** 0.5
print(f"LightGBM Validation RMSE: {rmse_lgb:.2f}")

#import xgboost as xgb
from sklearn.metrics import mean_squared_error

# ------------------------------
# 5. Train LightGBM
# ------------------------------

import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import numpy as np

# Define the model
lgb_model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    random_state=42
)

# Train the model with early stopping and logging
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='rmse',
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(50)]
)

# Predict on validation set
y_pred_lgb = lgb_model.predict(X_val)

# Compute RMSE manually (Python 3.12 compatible)
rmse_lgb = np.sqrt(mean_squared_error(y_val, y_pred_lgb))
print(f"LightGBM Validation RMSE: {rmse_lgb:.2f}")
# ------------------------------
# 5c. Train CatBoost
# ------------------------------
cat_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    random_state=42,
    verbose=50,
    early_stopping_rounds=50
)
cat_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val)
)

y_pred_cat = cat_model.predict(X_val)
rmse_cat = mean_squared_error(y_val, y_pred_cat) ** 0.5
print(f"CatBoost Validation RMSE: {rmse_cat:.2f}")

# ------------------------------
# 6. Generate Test Predictions (example LightGBM)
# ------------------------------
test['target_pred'] = lgb_model.predict(X_test)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000434 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2045
[LightGBM] [Info] Number of data points in the train set: 24445, number of used features: 9
[LightGBM] [Info] Start training from score 61.188992
Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 37.6886	valid_0's l2: 1420.43
[100]	valid_0's rmse: 36.8288	valid_0's l2: 1356.36
[150]	valid_0's rmse: 36.5811	valid_0's l2: 1338.18
[200]	valid_0's rmse: 36.451	valid_0's l2: 1328.67
[250]	valid_0's rmse: 36.3851	valid_0's l2: 1323.88
[300]	valid_0's rmse: 36.3175	valid_0's l2: 1318.96
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[350]	valid_0's rmse: 36.2508	valid_0's l2: 1314.12
[400]	valid_0's rmse: 36.215	valid_0's l2: 1311.53
[450]	valid_0's rmse: 36.1934	valid_0's l2: 1